# ML Zoomcamp 2026 — Homework 4
### Evaluation Metrics for Classification

**Dataset:** 2026 course lead scoring · **Target:** `converted`

This notebook solves all six questions in the [official Homework 4](https://github.com/DataTalksClub/machine-learning-zoomcamp/blob/main/cohorts/2026/homework/04-evaluation/homework.md). It follows the specified train/validation/test split (`random_state=1`), fits a logistic-regression classifier with `DictVectorizer`, and evaluates it using ROC AUC, Precision, Recall, F1, and 5-fold CV.

**Usage:** Run cells sequentially (`Run All`). If you have no internet in Jupyter, manually download `course_lead_scoring_2026.csv` into the same directory as this notebook. All outputs are computed when you run it; they are not hardcoded into code cells.

## 0. Imports and dataset preparation

- Categorical null values → `'NA'`
- Numerical null values → `0.0`
- Splitting: 80% full training / 20% test, then 75% of full training for training and 25% for validation (60/20/20 overall)

The 2026 pinned release contains **5,000 rows**. Do not use the similar-looking dataset from an earlier cohort.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, train_test_split

DATA_URL = (
    "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/"
    "main/cohorts/2026/data/course_lead_scoring_2026.csv"
)
DATA_FILE = Path("course_lead_scoring_2026.csv")

try:
    df = pd.read_csv(DATA_FILE if DATA_FILE.exists() else DATA_URL)
except Exception as exc:
    raise RuntimeError(
        f"Cannot load the dataset. Download it from {DATA_URL} "
        "and place it next to this notebook."
    ) from exc

assert len(df) == 5000, "Wrong dataset: the official 2026 release has 5,000 rows"
assert 'converted' in df.columns, 'Missing target column: converted'

categorical = df.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
numerical = [c for c in df.select_dtypes(include='number').columns if c != 'converted']
df[categorical] = df[categorical].fillna('NA')
df[numerical] = df[numerical].fillna(0.0)

print('Shape:', df.shape)
print('Categorical:', categorical)
print('Numerical:', numerical)
print('Missing values remaining:', df.isna().sum().sum())
display(df.head())

In [ ]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

TARGET = 'converted'
features = [col for col in df.columns if col != TARGET]
y_train = df_train[TARGET].to_numpy()
y_val = df_val[TARGET].to_numpy()

print('Train:', len(df_train), 'Validation:', len(df_val), 'Test:', len(df_test))
assert (len(df_train), len(df_val), len(df_test)) == (3000, 1000, 1000)

## Q1 — ROC AUC feature importance

Evaluate the **raw numerical features as scoring functions** on training data. If a feature produces AUC below `0.5`, negate it and compute AUC again. Choose the feature with the largest result out of `lead_score`, `number_of_courses_viewed`, `interaction_count`, `annual_income`.

In [ ]:
candidate_features = [
    'lead_score', 'number_of_courses_viewed',
    'interaction_count', 'annual_income',
]

q1_results = []
for column in candidate_features:
    values = df_train[column].to_numpy()
    auc = roc_auc_score(y_train, values)
    if auc < 0.5:
        auc = roc_auc_score(y_train, -values)
    q1_results.append({'feature': column, 'auc': auc})

q1_table = pd.DataFrame(q1_results).sort_values('auc', ascending=False)
display(q1_table.round(3))
q1_answer = q1_table.iloc[0]['feature']
print('Q1 ANSWER:', q1_answer)

## Q2 — Fit logistic regression; validation ROC AUC

One-hot encode **all feature columns** with `DictVectorizer`, excluding `converted`, and fit `LogisticRegression(solver='liblinear', C=1.0, max_iter=1000)`.

In [ ]:
dv = DictVectorizer(sparse=False)
train_dicts = df_train[features].to_dict(orient='records')
val_dicts = df_val[features].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000)
model.fit(X_train, y_train)

val_pred_proba = model.predict_proba(X_val)[:, 1]
q2_answer = roc_auc_score(y_val, val_pred_proba)
print(f'Q2 ANSWER: {q2_answer:.3f}')

## Q3 — Precision / Recall intersection

Calculate TP, FP and FN at every threshold from **0.00** through **1.00**, with a step of **0.01**. Exclude points where both Precision and Recall equal zero, then find the smallest absolute distance between them.

In [ ]:
thresholds = np.round(np.arange(0.0, 1.001, 0.01), 2)
threshold_rows = []

for threshold in thresholds:
    is_positive = val_pred_proba >= threshold
    tp = np.sum(is_positive & (y_val == 1))
    fp = np.sum(is_positive & (y_val == 0))
    fn = np.sum(~is_positive & (y_val == 1))

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    threshold_rows.append((threshold, precision, recall, tp, fp, fn))

pr = pd.DataFrame(threshold_rows, columns=[
    'threshold', 'precision', 'recall', 'tp', 'fp', 'fn'
])
valid = pr.loc[(pr['precision'] > 0) | (pr['recall'] > 0)].copy()
valid['abs_difference'] = (valid['precision'] - valid['recall']).abs()
q3_row = valid.loc[valid['abs_difference'].idxmin()]
q3_answer = float(q3_row['threshold'])

fig, ax = plt.subplots(figsize=(9, 4.8))
ax.plot(pr['threshold'], pr['precision'], label='Precision')
ax.plot(pr['threshold'], pr['recall'], label='Recall')
ax.axvline(q3_answer, color='black', linestyle='--', alpha=.6,
           label=f'Closest point: {q3_answer:.2f}')
ax.set(xlabel='Probability threshold', ylabel='Score',
       title='Precision and Recall vs. threshold', xlim=(0, 1), ylim=(0, 1.02))
ax.legend(); ax.grid(alpha=.2)
plt.show()
print('Q3 ANSWER:', f'{q3_answer:.2f}')
display(q3_row[['threshold', 'precision', 'recall', 'abs_difference']].to_frame().T.round(5))

## Q4 — Maximum F1 score

At each threshold, compute the harmonic mean: `F1 = 2 * precision * recall / (precision + recall)`, using 0 when the denominator is 0.

In [ ]:
pr['f1'] = [
    2 * p * r / (p + r) if (p + r) else 0.0
    for p, r in zip(pr['precision'], pr['recall'])
]
q4_row = pr.loc[pr['f1'].idxmax()]
q4_answer = float(q4_row['threshold'])

fig, ax = plt.subplots(figsize=(9, 4.8))
ax.plot(pr['threshold'], pr['f1'], label='F1')
ax.axvline(q4_answer, color='black', linestyle='--', alpha=.6,
           label=f'Maximum F1 threshold: {q4_answer:.2f}')
ax.set(xlabel='Probability threshold', ylabel='F1 score',
       title='F1 score vs. threshold', xlim=(0, 1))
ax.legend(); ax.grid(alpha=.2)
plt.show()
print(f'Q4 ANSWER: {q4_answer:.2f} (F1={q4_row["f1"]:.4f})')

## Q5 — 5-fold cross-validation

Use the entire `df_full_train`, `KFold(n_splits=5, shuffle=True, random_state=1)`, and the same logistic-regression parameters as Q2.

**Important:** re-fit `DictVectorizer` in every fold on the training partition only; do not fit it on the validation fold.

In [ ]:
def cross_validate_auc(data: pd.DataFrame, C: float = 1.0) -> np.ndarray:
    kfold = KFold(n_splits=5, shuffle=True, random_state=1)
    aucs = []

    for train_idx, val_idx in kfold.split(data):
        fold_train = data.iloc[train_idx]
        fold_val = data.iloc[val_idx]
        y_fold_train = fold_train[TARGET].to_numpy()
        y_fold_val = fold_val[TARGET].to_numpy()

        vectorizer = DictVectorizer(sparse=False)
        X_fold_train = vectorizer.fit_transform(
            fold_train[features].to_dict(orient='records')
        )
        X_fold_val = vectorizer.transform(
            fold_val[features].to_dict(orient='records')
        )

        estimator = LogisticRegression(solver='liblinear', C=C, max_iter=1000)
        estimator.fit(X_fold_train, y_fold_train)
        probabilities = estimator.predict_proba(X_fold_val)[:, 1]
        aucs.append(roc_auc_score(y_fold_val, probabilities))

    return np.array(aucs)

q5_scores = cross_validate_auc(df_full_train, C=1.0)
q5_answer = np.std(q5_scores)
print('Fold AUCs:', np.round(q5_scores, 4))
print(f'Q5 ANSWER: {q5_answer:.3f}')

## Q6 — Hyperparameter tuning with 5-fold CV

Try `C = 0.000001`, `C = 0.001`, `C = 1`. Rank by **rounded mean AUC (highest)**, **rounded std (lowest)**, and finally **C (smallest)** to apply the official tie-breaking rules.

In [ ]:
parameter_rows = []
for C in [0.000001, 0.001, 1]:
    fold_scores = cross_validate_auc(df_full_train, C=C)
    parameter_rows.append({
        'C': C,
        'mean_auc': np.mean(fold_scores),
        'std_auc': np.std(fold_scores),
    })

q6_table = pd.DataFrame(parameter_rows)
q6_table['mean_rounded'] = q6_table['mean_auc'].round(3)
q6_table['std_rounded'] = q6_table['std_auc'].round(3)
q6_ranking = q6_table.sort_values(
    ['mean_rounded', 'std_rounded', 'C'], ascending=[False, True, True]
)
q6_answer = float(q6_ranking.iloc[0]['C'])
display(q6_table[['C', 'mean_rounded', 'std_rounded']])
print(f'Q6 ANSWER: C={q6_answer:g}')

## All six answers

This summary is **generated from the calculations above** (not embedded or pre-filled).

In [ ]:
final_answers = pd.DataFrame({
    'Question': ['Q1', 'Q2', 'Q3', 'Q4', 'Q5', 'Q6'],
    'Answer': [
        q1_answer, f'{q2_answer:.3f}', f'{q3_answer:.2f}',
        f'{q4_answer:.2f}', f'{q5_answer:.3f}', f'{q6_answer:g}'
    ]
})
display(final_answers)

### Reference values / expected options

As a cross-check, a publicly executed [2026 notebook](https://raw.githubusercontent.com/AnnalieseTech/2026_ML_ZOOMCAMP/main/04_evaluation/04_hw.ipynb) reports:

| Question | Expected answer |
|:--|:--|
| Q1 | `lead_score` (AUC about 0.788) |
| Q2 | `0.732` |
| Q3 | `0.63` |
| Q4 | `0.41` |
| Q5 | `0.007` |
| Q6 | `0.001` (mean AUC 0.749; std 0.010) |

The cells above perform each calculation independently when executed; they have not been run in this generated notebook. Differences from these values usually indicate a different 2025 dataset, an incorrect random seed, a target included among the features, or a different missing-value strategy.

**Homework submission:** [Course platform — HW04](https://courses.datatalks.club/ml-zoomcamp-2026/homework/hw04).